In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [17]:
df = pd.read_csv("dataset.csv")
df.head()
df2 = df[df['track_genre'] == 'r-n-b']
df2.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
87000,87000,3J8EOeKLTLXORtWPpOU5bE,Lil Tjay;6LACK,Destined 2 Win,Calling My Phone,81,205458,True,0.907,0.393,...,-7.636,0,0.0539,0.451,0.000001,0.1350,0.202,104.949,4,r-n-b
87001,87001,5JtvedSVWW9McnoEAPJwQm,Lil Tjay,True 2 Myself,Sex Sounds,76,162453,True,0.857,0.427,...,-8.638,1,0.1120,0.281,0.000000,0.1080,0.366,130.033,4,r-n-b
87002,87002,3SdTKo2uVsxFblQjpScoHy,Ben E. King,Don't Play That Song (Mono),Stand by Me,80,180055,False,0.650,0.306,...,-9.443,1,0.0393,0.570,0.000007,0.0707,0.605,118.068,4,r-n-b
87003,87003,2DpJ9T2RVRanZcYFHKOAfA,Paul Anka,Put Your Head On My Shoulder: The Very Best Of...,Put Your Head On My Shoulder,69,154733,False,0.547,0.335,...,-9.200,0,0.0307,0.747,0.000000,0.1010,0.474,115.838,3,r-n-b
87004,87004,75FEaRjZTKLhTrFGsfMUXR,Kate Bush,Hounds Of Love,Running Up That Hill (A Deal With God),90,298933,False,0.629,0.547,...,-13.123,0,0.0550,0.720,0.003140,0.0604,0.197,108.375,4,r-n-b


In [ ]:
def get_music_recommendations(df, track_ids, n_recommendations=5, aggregation_method='average'):
    """
    Get music recommendations for multiple tracks using KNN with aggregated similarity scores.

    Params:
    df
    track_id
    n_recommendations
    aggregation_method: how to combine scores from multiple input tracks ('average', 'max', or 'centroid')

    Returns:
    A list of dictionaries containing recommendation details with aggregated similarity scores
    """
    features = ['popularity', 'danceability', 'energy', 'loudness',
               'speechiness', 'acousticness', 'instrumentalness',
               'liveness', 'valence', 'tempo']

    feature_matrix = df[features].values
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(feature_matrix)

    if aggregation_method == 'centroid':
        # Calculate centroid of input tracks
        input_indices = [df[df['track_id'] == track_id].index[0] for track_id in track_ids]
        centroid = features_scaled[input_indices].mean(axis=0).reshape(1, -1)
        
        knn = NearestNeighbors(n_neighbors=n_recommendations + len(track_ids), metric='euclidean')
        knn.fit(features_scaled)
        distances, indices = knn.kneighbors(centroid)
        
        # Filter out input tracks from recommendations
        mask = ~np.isin(indices.flatten(), input_indices)
        indices = indices.flatten()[mask][:n_recommendations]
        distances = distances.flatten()[mask][:n_recommendations]
        
        recommendations = []
        for idx, distance in zip(indices, distances):
            track = df.iloc[idx]
            similarity_score = 1 / (1 + distance)
            recommendations.append({
                'track_id': track['track_id'],
                'track_name': track['track_name'],
                'artist': track['artists'],
                'album': track['album_name'],
                'genre': track['track_genre'],
                'similarity_score': similarity_score,
                'input_similarities': {tid: similarity_score for tid in track_ids}
            })
        
        return recommendations

    else:  # 'average' or 'max' aggregation
        knn = NearestNeighbors(n_neighbors=n_recommendations + 1, metric='euclidean')
        knn.fit(features_scaled)

        recommendations_dict = {}
        
        for track_id in track_ids:
            track_idx = df[df['track_id'] == track_id].index[0]
            distances, indices = knn.kneighbors(
                features_scaled[track_idx].reshape(1, -1)
            )
            
            indices = indices.flatten()[1:]
            distances = distances.flatten()[1:]
            
            for idx, distance in zip(indices, distances):
                track = df.iloc[idx]
                rec_id = track['track_id']
                similarity_score = 1 / (1 + distance)
                
                if rec_id not in recommendations_dict:
                    recommendations_dict[rec_id] = {
                        'track_id': rec_id,
                        'track_name': track['track_name'],
                        'artist': track['artists'],
                        'album': track['album_name'],
                        'genre': track['track_genre'],
                        'input_similarities': {}
                    }
                recommendations_dict[rec_id]['input_similarities'][track_id] = similarity_score
        recommendations = []
        for rec in recommendations_dict.values():
            if aggregation_method == 'average':
                rec['similarity_score'] = np.mean(list(rec['input_similarities'].values()))
            else:  # 'max'
                rec['similarity_score'] = max(rec['input_similarities'].values())
            recommendations.append(rec)

        recommendations.sort(key=lambda x: -x['similarity_score'])
        return recommendations[:n_recommendations]

def print_recommendations(df, track_ids, n_recommendations=5, aggregation_method='average'):
    """
    Format recommendations with detailed similarity information
    """
    recommendations = get_music_recommendations(
        df, track_ids, n_recommendations, aggregation_method
    )

    print(f"\nTop {n_recommendations} Recommended tracks (using {aggregation_method} aggregation):")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. {rec['track_name']} by {rec['artist']}")
        print(f"   Album: {rec['album']}")
        print(f"   Genre: {rec['genre']}")
        print(f"   Overall Similarity Score: {rec['similarity_score']:.3f}")
        print("   Individual Similarities:")
        for track_id, score in rec['input_similarities'].items():
            input_track = df[df['track_id'] == track_id].iloc[0]
            print(f"     - {input_track['track_name']}: {score:.3f}")

In [35]:
track_ids = ['5SuOikwiRyPMVoIQDJUgSV', '4qPNDBW1i3p13qLCt0Ki3A', '1iJBSr7s7jYXzM8EGcbK5b']
print_recommendations(df, track_ids)

Top 5 Recommended tracks:

1. Comedy by Gen Hoshino
   Album: Comedy
   Genre: acoustic
   Similarity Score: 1.000

2. Comedy by Gen Hoshino
   Album: Comedy
   Genre: singer-songwriter
   Similarity Score: 1.000

3. Comedy by Gen Hoshino
   Album: Comedy
   Genre: songwriter
   Similarity Score: 1.000

4. Ghost - Acoustic by Ben Woodward
   Album: Ghost (Acoustic)
   Genre: acoustic
   Similarity Score: 1.000

5. Feel Again (Feat. Au/Ra) by Kina;Au/Ra
   Album: Things I Wanted To Tell You
   Genre: sad
   Similarity Score: 0.642


In [36]:
# 2 r-n-b and 1 acoustic
track_ids = ['3J8EOeKLTLXORtWPpOU5bE', '5JtvedSVWW9McnoEAPJwQm', '1iJBSr7s7jYXzM8EGcbK5b']
print_recommendations(df, track_ids)

Top 5 Recommended tracks:

1. Feel Again (Feat. Au/Ra) by Kina;Au/Ra
   Album: Things I Wanted To Tell You
   Genre: sad
   Similarity Score: 0.642

2. Atlantis - Slowed Down Version by Seafret
   Album: Atlantis
   Genre: folk
   Similarity Score: 0.583

3. You & I (Nobody in the World) by John Legend
   Album: Love In The Future (Expanded Edition)
   Genre: soul
   Similarity Score: 0.576

4. The Blood by Bethel Music;Jenn Johnson;Mitch Wong
   Album: Simple
   Genre: world-music
   Similarity Score: 0.572

5. The Blood by Bethel Music;Jenn Johnson;Mitch Wong
   Album: Simple
   Genre: ambient
   Similarity Score: 0.572
